In [0]:
from pyspark.sql import Row

worker_data = [
    Row(worker_id=1, first_name="John", last_name="Doe", salary=5000, joining_date="2023-01-01", department="Engineering"),
    Row(worker_id=2, first_name="Jane", last_name="Smith", salary=6000, joining_date="2023-01-15", department="Marketing"),
    Row(worker_id=3, first_name="Alice", last_name="Johnson", salary=4500, joining_date="2023-02-05", department="Engineering")
]

title_data = [
    Row(worker_ref_id=1, worker_title="Engineer", affected_from="2022-01-01"),
    Row(worker_ref_id=2, worker_title="Manager", affected_from="2022-01-01"),
    Row(worker_ref_id=3, worker_title="Engineer", affected_from="2022-01-01")
]

worker_df = spark.createDataFrame(worker_data)
title_df = spark.createDataFrame(title_data)

display(worker_df)
display(title_df)

In [0]:
from pyspark.sql.functions import *

high_sal = worker_df.agg(max(col('salary'))).collect()[0][0]
low_sal = worker_df.agg(min(col('salary'))).collect()[0][0]

In [0]:
whdf = worker_df.filter(col('salary')==high_sal).withColumn('salary_type',lit('highest_salary'))
wldf = worker_df.filter(col('salary')==low_sal).withColumn('salary_type',lit('lowest_salary'))

total_df = whdf.union(wldf)
total_df.show()


best approach as per chag gpt


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import max, min, when, col

w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

result_df = (
    worker_df
    .withColumn("max_salary", max("salary").over(w))
    .withColumn("min_salary", min("salary").over(w))
    .withColumn(
        "salary_type",
        when(col("salary") == col("max_salary"), "Highest Salary")
        .when(col("salary") == col("min_salary"), "Lowest Salary")
    )
    .filter(col("salary_type").isNotNull())
    .drop("max_salary", "min_salary")
    .join(title_df, worker_df.worker_id == title_df.worker_ref_id, "left")
    .select(
        "worker_id", "first_name", "last_name",
        "salary", "salary_type", "worker_title", "department"
    )
)

result_df.show()
